# Ripon Grammar School — Admissions Boundary Analysis

**Goal**: Estimate where the out-of-catchment admissions boundary (maximum day-place distance) is likely to be in future years, to help assess the probability of a place being offered to a child living outside the defined catchment area.

**Key facts**
- PAN: 117 (including 14 boarding places) → ~103 day places
- Admissions criteria order: Looked-after children → siblings → **in-catchment** → out-of-catchment by distance
- The `distance of last child allocated` reported each year is always for an **out-of-catchment** day pupil — all in-catchment applicants receive offers
- Data covers entry years 2013–2025 (12 cohorts); 2013-14 applications column is unreliable

**Data sources**
1. North Yorkshire secondary admissions statistics (Excel, 2013-14 to 2025-26)
2. ONS live births by local authority 1992-2021 — Harrogate district (E07000165)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

## 1. Load data

In [ ]:
# Load the pre-built merged CSV (admissions + Harrogate births)
df = pd.read_csv('../data/processed/rgs_with_births.csv')

# 2013-14 applications figure is 0 in source (data quality issue with that year's format)
# Drop from analysis where noted
df.loc[df['entry_year'] == 2013, 'applications'] = np.nan

print(f'Rows: {len(df)}')
df[['year_key','entry_year','birth_year','total_births',
    'applications','allocated','day_distance_miles','waiting_list']]

## 2. Time-series overview

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)

# --- Distance ---
ax = axes[0]
ax.plot(df['entry_year'], df['day_distance_miles'], 'o-', color='steelblue', ms=6)
# Flag the 2022-23 outlier
outlier = df[df['entry_year'] == 2022]
ax.scatter(outlier['entry_year'], outlier['day_distance_miles'],
           color='red', zorder=5, s=80, label='Possible anomaly (2022-23)')
ax.set_ylabel('Day-place boundary (miles)')
ax.set_title('Distance of last out-of-catchment day pupil allocated')
ax.legend(fontsize=8)
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())

# --- Applications ---
ax = axes[1]
ax.bar(df['entry_year'], df['applications'], color='teal', alpha=0.7)
ax.set_ylabel('Applications (all preferences)')
ax.set_title('Total applications received')

# --- Harrogate births (shifted to entry year) ---
ax = axes[2]
ax.bar(df['entry_year'], df['total_births'], color='darkorange', alpha=0.7)
ax.set_ylabel('Total births')
ax.set_title('Harrogate district births (cohort 11 years prior to entry)')
ax.set_xlabel('Entry year (September)')

for ax in axes:
    ax.set_xticks(df['entry_year'])
    ax.set_xticklabels(df['year_key'], rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig('../data/processed/fig_timeseries.png', bbox_inches='tight')
plt.show()

### 2.1 Observations

- The boundary fluctuates widely: **6.6 – 20.4 miles** across 11 years with data.
- **2022-23 (20.36 miles)** is a clear outlier despite having below-average applications and a normal birth cohort. Possible explanations: unusual in-catchment take-up that year (leaving more out-of-catchment slots), or a data entry error. This year should be treated with caution in modelling.
- **Applications** nearly tripled in 2019-20 (477) — likely reflecting post-Ofsted or word-of-mouth interest — before falling back. They are rising again in 2024-25.
- **Birth cohort size** in Harrogate peaked around 2007-08 (≈1,700) and has been declining: the 2014 cohort (entering 2025) is at ≈1,437. This declining pool should ease competition slightly.
- The ONS birth series ends at 2021, so cohorts entering from **2032 onwards** will need ONS projections.

## 3. Correlation analysis

In [ ]:
# Exclude 2013-14 (no applications) and the 2022-23 outlier for modelling
df_clean = df[(df['entry_year'] > 2013) & (df['day_distance_miles'].notna())].copy()
df_model = df_clean[df_clean['entry_year'] != 2022].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Distance vs applications
ax = axes[0]
ax.scatter(df_model['applications'], df_model['day_distance_miles'],
           color='steelblue', s=60, zorder=3, label='Model data')
ax.scatter(df_clean[df_clean['entry_year']==2022]['applications'],
           df_clean[df_clean['entry_year']==2022]['day_distance_miles'],
           color='red', s=80, marker='x', linewidths=2, label='2022-23 outlier')
for _, row in df_model.iterrows():
    ax.annotate(str(int(row['entry_year'])), (row['applications'], row['day_distance_miles']),
                textcoords='offset points', xytext=(4,4), fontsize=7)
m, b, r, p, _ = stats.linregress(df_model['applications'], df_model['day_distance_miles'])
xs = np.linspace(df_model['applications'].min(), df_model['applications'].max(), 100)
ax.plot(xs, m*xs + b, '--', color='steelblue', alpha=0.6)
ax.set_xlabel('Applications (all preferences)')
ax.set_ylabel('Day boundary (miles)')
ax.set_title(f'Distance vs Applications\nr = {r:.2f}, p = {p:.3f}')
ax.legend(fontsize=8)

# Distance vs births
ax = axes[1]
ax.scatter(df_model['total_births'], df_model['day_distance_miles'],
           color='darkorange', s=60, zorder=3, label='Model data')
ax.scatter(df_clean[df_clean['entry_year']==2022]['total_births'],
           df_clean[df_clean['entry_year']==2022]['day_distance_miles'],
           color='red', s=80, marker='x', linewidths=2, label='2022-23 outlier')
for _, row in df_model.iterrows():
    ax.annotate(str(int(row['entry_year'])), (row['total_births'], row['day_distance_miles']),
                textcoords='offset points', xytext=(4,4), fontsize=7)
m2, b2, r2, p2, _ = stats.linregress(df_model['total_births'], df_model['day_distance_miles'])
xs2 = np.linspace(df_model['total_births'].min(), df_model['total_births'].max(), 100)
ax.plot(xs2, m2*xs2 + b2, '--', color='darkorange', alpha=0.6)
ax.set_xlabel('Harrogate births (11 years prior)')
ax.set_ylabel('Day boundary (miles)')
ax.set_title(f'Distance vs Births\nr = {r2:.2f}, p = {p2:.3f}')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/processed/fig_correlations.png', bbox_inches='tight')
plt.show()

print(f'Applications → distance:  r = {r:.3f}, p = {p:.4f}')
print(f'Births → distance:        r = {r2:.3f}, p = {p2:.4f}')

## 4. Simple regression model

We fit two OLS models:
- **Model A**: `distance ~ applications` (single predictor, most directly observable)
- **Model B**: `distance ~ total_births + applications` (two predictors)

With only ~10 usable data points (excluding 2013-14 and the 2022-23 outlier) these are illustrative rather than definitive.

In [ ]:
from scipy.stats import t as t_dist

def ols_summary(X: np.ndarray, y: np.ndarray, feature_names: list[str]) -> dict:
    """Minimal OLS: returns coefficients, R², and 95% prediction interval width."""
    n, k = X.shape
    Xb = np.hstack([np.ones((n,1)), X])
    beta = np.linalg.lstsq(Xb, y, rcond=None)[0]
    y_hat = Xb @ beta
    resid = y - y_hat
    ss_res = (resid**2).sum()
    ss_tot = ((y - y.mean())**2).sum()
    r2 = 1 - ss_res/ss_tot
    s2 = ss_res / (n - k - 1)
    return dict(beta=beta, r2=r2, s=np.sqrt(s2), Xb=Xb, y_hat=y_hat, n=n, k=k)

def predict_pi(model: dict, x_new: np.ndarray, alpha=0.05) -> tuple[float, float, float]:
    """Return (point estimate, lower 95% PI, upper 95% PI) for a new observation."""
    n, k = model['n'], model['k']
    x_new_b = np.array([1.0] + list(x_new))
    y_hat = float(x_new_b @ model['beta'])
    XtXinv = np.linalg.inv(model['Xb'].T @ model['Xb'])
    leverage = float(x_new_b @ XtXinv @ x_new_b)
    se_pred = model['s'] * np.sqrt(1 + leverage)
    t_crit = t_dist.ppf(1 - alpha/2, df=n - k - 1)
    return y_hat, y_hat - t_crit * se_pred, y_hat + t_crit * se_pred

# Prepare arrays (drop rows with NaN in applications or distance)
dm = df_model.dropna(subset=['applications','total_births','day_distance_miles'])
y = dm['day_distance_miles'].values
X_a = dm[['applications']].values
X_b = dm[['total_births','applications']].values

model_a = ols_summary(X_a, y, ['applications'])
model_b = ols_summary(X_b, y, ['total_births','applications'])

print(f'Model A (applications only):          R² = {model_a["r2"]:.3f},  residual std = {model_a["s"]:.2f} miles')
print(f'Model B (births + applications):      R² = {model_b["r2"]:.3f},  residual std = {model_b["s"]:.2f} miles')
print()
print('Model A coefficients: intercept =', round(model_a['beta'][0],3),
      '  applications =', round(model_a['beta'][1],4))
print('Model B coefficients: intercept =', round(model_b['beta'][0],3),
      '  births =', round(model_b['beta'][1],4),
      '  applications =', round(model_b['beta'][2],4))

## 5. Forward projections

**ONS birth data limitation**: The ONS series ends at 2021, giving us Harrogate births through 2021 (→ entry year 2032). For cohorts entering 2026-2031 we use actual births; for 2032+ we would need ONS sub-national population projections.

We scenario-test three application levels (low / central / high) reflecting observed range (200–350 excluding the 2019-20 outlier of 477).

In [ ]:
births_df = pd.read_csv('../data/processed/ons_births_harrogate.csv')
births_df['entry_year'] = births_df['birth_year'] + 11

# Future entry years we can project (ONS births available through 2021 → entry 2032)
future_years = list(range(2026, 2033))
future_births = births_df[births_df['entry_year'].isin(future_years)][['entry_year','total_births']]

# Three application scenarios
scenarios = {'Low (200)': 200, 'Central (270)': 270, 'High (330)': 330}

rows = []
for _, fb in future_births.iterrows():
    ey = int(fb['entry_year'])
    tb = fb['total_births']
    for scenario, apps in scenarios.items():
        est_a, lo_a, hi_a = predict_pi(model_a, [apps])
        est_b, lo_b, hi_b = predict_pi(model_b, [tb, apps])
        rows.append(dict(entry_year=ey, total_births=tb, scenario=scenario,
                         apps_assumed=apps,
                         model_a_est=round(est_a,2), model_a_lo=round(lo_a,2), model_a_hi=round(hi_a,2),
                         model_b_est=round(est_b,2), model_b_lo=round(lo_b,2), model_b_hi=round(hi_b,2)))

proj = pd.DataFrame(rows)
print(proj[['entry_year','total_births','scenario','apps_assumed',
            'model_a_est','model_a_lo','model_a_hi',
            'model_b_est','model_b_lo','model_b_hi']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
colours = {'Low (200)': 'C0', 'Central (270)': 'C1', 'High (330)': 'C2'}

for ax_idx, (model_label, est_col, lo_col, hi_col) in enumerate([
    ('Model A — applications only', 'model_a_est', 'model_a_lo', 'model_a_hi'),
    ('Model B — births + applications', 'model_b_est', 'model_b_lo', 'model_b_hi'),
]):
    ax = axes[ax_idx]

    # Historical actuals
    ax.plot(df_clean[df_clean['entry_year']!=2022]['entry_year'],
            df_clean[df_clean['entry_year']!=2022]['day_distance_miles'],
            'ko-', ms=5, label='Historical (used in fit)')
    ax.plot(df_clean[df_clean['entry_year']==2022]['entry_year'],
            df_clean[df_clean['entry_year']==2022]['day_distance_miles'],
            'rx', ms=10, mew=2, label='2022-23 (excluded from fit)')

    for scenario, colour in colours.items():
        sub = proj[proj['scenario'] == scenario].sort_values('entry_year')
        ax.plot(sub['entry_year'], sub[est_col], 'o--', color=colour, label=f'{scenario}')
        ax.fill_between(sub['entry_year'], sub[lo_col], sub[hi_col],
                        color=colour, alpha=0.12)

    ax.axhline(0, color='grey', lw=0.5, ls=':')
    ax.set_title(model_label)
    ax.set_xlabel('Entry year')
    ax.set_ylabel('Day boundary (miles)')
    ax.legend(fontsize=8)
    ax.set_xlim(2013, 2033)

plt.suptitle('RGS out-of-catchment day boundary — historical and projected', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/fig_projections.png', bbox_inches='tight')
plt.show()

## 6. Summary & caveats

### What the model says
- Excluding the 2022-23 outlier, the **central projection** for near-term entry years is roughly **8–12 miles**, with wide 95% prediction intervals of ±5–8 miles given the limited sample.
- The **declining birth cohort** in Harrogate (≈1,700 in 2007 down to ≈1,437 in 2014) is a mild tailwind — smaller pools tend to mean slightly wider accessible boundaries.
- **Applications** are the stronger predictor of the boundary than births alone, but they are only known after the application cycle closes each January — making them a nowcast variable rather than a true forecast.

### Important unknowns that would sharpen the model
| Gap | Impact | How to fill |
|-----|--------|-------------|
| In-catchment vs out-of-catchment split each year | High — determines how many day slots reach out-of-catchment | FOI request to North Yorkshire admissions |
| Actual number of first-preference out-of-catchment applicants | High — 'all preferences' includes safety-net applications | FOI / admissions team |
| Why 2022-23 was 20.36 miles | Medium — affects whether to include it | North Yorkshire admissions data |
| ONS births 2022+ (→ entry years 2033+) | Low for near term | ONS sub-national population projections |
| Your distance from RGS | Critical for the actual decision | Measure straight-line or road distance |

### Next step for your specific situation
1. **Measure your distance** to RGS (straight-line and road distance — confirm which the school uses).
2. **Check the admissions policy** on the RGS website for the exact distance-measurement method.
3. Compare your distance to the historical range (6.6 – 13.7 miles, excluding outliers). If you are within 7 miles you are likely safe; 8–11 miles is uncertain; beyond 12 miles is risky based on history.